In [1]:
# ==========================================
# IMPORTURI ȘI CURĂȚARE MEMORIE
# ==========================================
import os
import time
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# Asigurăm-ne că memoria e goală la început
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 0. PLANUL DE LUPTĂ (LISTA DE EXPERIMENTE)
# ==========================================
EXPERIMENTE = [
    
    # 1. Combinate (fără gigantul EyePACS), 50 epoci, cu augmentare
    {"nume": "Exp1_ResNet50_Combined3_Aug", "model": "resnet50", "datasets": ["APTOS", "IDRID", "MESSIDOR"], "epoci": 50, "aug": True},
    {"nume": "Exp2_DenseNet121_Combined3_Aug", "model": "densenet121", "datasets": ["APTOS", "IDRID", "MESSIDOR"], "epoci": 50, "aug": True},
    {"nume": "Exp3_EfficientNetB0_Combined3_Aug", "model": "efficientnet_b0", "datasets": ["APTOS", "IDRID", "MESSIDOR"], "epoci": 50, "aug": True},

    # 2. Toate combinate random, 15 epoci, fără augmentare
    {"nume": "Exp4_ResNet50_ALL_Combined", "model": "resnet50", "datasets": ["APTOS", "IDRID", "MESSIDOR", "EYEPACS"], "epoci": 15, "aug": False},

    # 3. Fără augmentare, EyePACS, 20 epoci
    {"nume": "Exp5_ResNet50_EyePACS", "model": "resnet50", "datasets": ["EYEPACS"], "epoci": 20, "aug": False},
    {"nume": "Exp6_DenseNet121_EyePACS", "model": "densenet121", "datasets": ["EYEPACS"], "epoci": 20, "aug": False},
    {"nume": "Exp7_EfficientNetB0_EyePACS", "model": "efficientnet_b0", "datasets": ["EYEPACS"], "epoci": 20, "aug": False}
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rulăm pe: {device} | GPU-uri paralele: {torch.cuda.device_count()}")

# ==========================================
# 1. MANAGER SETURI DE DATE ȘI PATH-URI
# ==========================================
def get_dataset_paths(dataset_name):
    """ ATENȚIE: Modifică base_path pentru IDRID, MESSIDOR, EYEPACS cu folderele tale reale! """
    if dataset_name == "APTOS":
        # base = "/kaggle/input/datasets/mariaherrerot/aptos2019" 
        base = "/home/marian-s/Disertatie/Dataset_APTOS_2019"
        return {"csv": f"{base}/train_1.csv", "dir": f"{base}/train_images/train_images"}
        
    elif dataset_name == "EYEPACS":
        # base = "/kaggle/input/datasets/c7934597/resized-2015-2019-diabetic-retinopathy-detection" # <-- VERIFICĂ NUMELE
        base = "/home/marian-s/Disertatie/Dataset_KaggleEyePACS"
        return {"csv": f"{base}/labels/traintestLabels15_trainLabels19.csv", "dir": f"{base}/resized_traintest15_train19"}
        
    elif dataset_name == "IDRID":
        # base = "/kaggle/input/datasets/mariaherrerot/idrid-dataset" # <-- VERIFICĂ NUMELE
        base = "/home/marian-s/Disertatie/Dataset_IDRID"
        return {"csv": f"{base}/idrid_labels.csv", "dir": f"{base}/Imagenes/Imagenes"}
        
    elif dataset_name == "MESSIDOR":
        # base = "/kaggle/input/datasets/mariaherrerot/messidor2preprocess" # <-- VERIFICĂ NUMELE
        base = "/home/marian-s/Disertatie/Dataset_Messidor_2"
        return {"csv": f"{base}/messidor_data.csv", "dir": f"{base}/messidor-2/messidor-2/preprocess"}
        
    else:
        raise ValueError(f"Dataset necunoscut: {dataset_name}")

# ==========================================
# 2. DATASET COMBINAT (COREZOLVAREA EXTENSIILOR)
# ==========================================
class CombinedRetinopathyDataset(Dataset):
    def __init__(self, dataset_names, transform=None):
        self.transform = transform
        self.samples = [] 
        
        for ds_name in dataset_names:
            paths = get_dataset_paths(ds_name)
            df = pd.read_csv(paths["csv"])
            
            for idx in range(len(df)):
                img_name = str(df.iloc[idx, 0])
                
                # 1. Curățăm complet numele de orice extensie veche din CSV (ex: scoatem .JPG)
                base_name = os.path.splitext(img_name)[0]
                
                # 2. Forțăm extensia corectă în funcție de setul de date
                if ds_name == "EYEPACS":
                    nume_final = base_name + '.jpeg'
                elif ds_name == "APTOS":
                    nume_final = base_name + '.png'
                elif ds_name == "MESSIDOR":
                    nume_final = base_name + '.png'  # <-- Aici am forțat formatul PNG pentru Messidor
                elif ds_name == "IDRID":
                    nume_final = base_name + '.jpg'
                else:
                    nume_final = img_name
                    
                full_path = os.path.join(paths["dir"], nume_final)
                
                # Transformare în problemă binară (0 = Normal, restul = Bolnav)
                label_initial = int(df.iloc[idx, 1])
                label_binar = 0 if label_initial == 0 else 1
                
                self.samples.append({'path': full_path, 'label': label_binar})
        
        print(f"  -> Bază de date încărcată: {len(self.samples)} imagini totale.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        path = item['path']
        
        # 3. Mecanismul de siguranță suprem
        if not os.path.exists(path):
            # Dacă nu găsește fișierul, luăm doar numele de bază și încercăm toate variantele
            baza = os.path.splitext(path)[0]
            for ext in ['.png', '.PNG', '.jpg', '.JPG', '.jpeg', '.JPEG']:
                if os.path.exists(baza + ext):
                    path = baza + ext
                    break

        # 4. Încărcare imagine cu bloc de protecție (Try/Except)
        try:
            image = Image.open(path).convert('RGB')
        except FileNotFoundError:
            # Dacă absolut nicio extensie nu a mers (poate lipsește o poză din arhivă), 
            # generăm o imagine neagră neutră pentru a nu opri antrenamentul.
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            # Opțional: print(f"Atenție, lipsește imaginea: {path}")
            
        label = torch.tensor(item['label'], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        return image, label

# ==========================================
# 3. SELECTOR DE MODELE
# ==========================================
def build_model(model_name):
    if model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        num_ftrs = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_ftrs, 1))
        return model, model.fc
        
    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        num_ftrs = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_ftrs, 1))
        return model, model.classifier
        
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        num_ftrs = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(p=0.5, inplace=True), nn.Linear(num_ftrs, 1))
        return model, model.classifier

# ==========================================
# 4. ORCHESTRATORUL (RULAREA AUTOMATĂ)
# ==========================================
for idx_exp, exp in enumerate(EXPERIMENTE):
    print(f"\n{'='*60}")
    print(f"🚀 ÎNCEPE EXPERIMENTUL {idx_exp + 1}/{len(EXPERIMENTE)}: {exp['nume']}")
    print(f"{'='*60}")
    
    # 4.1 Preprocesare
    transformari_lista = [transforms.Resize((224, 224))]
    if exp["aug"]:
        transformari_lista.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2)
        ])
    transformari_lista.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transforms_pipeline = transforms.Compose(transformari_lista)
    
    # 4.2 Încărcare Date
    full_dataset = CombinedRetinopathyDataset(exp["datasets"], transform=transforms_pipeline)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_ds, test_ds = torch.utils.data.random_split(full_dataset, [train_size, test_size])
    
    # MODIFICARE CRITICĂ: num_workers redus la 2 pentru a salva RAM
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    
    # 4.3 Inițializare Model
    model, strat_antrenat = build_model(exp["model"])
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    if isinstance(model, nn.DataParallel):
        if exp["model"] == "resnet50": params_to_train = model.module.fc.parameters()
        else: params_to_train = model.module.classifier.parameters()
    else:
        params_to_train = strat_antrenat.parameters()
        
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(params_to_train, lr=0.001)
    
    # 4.4 Bucla de Antrenare
    istoric = {'train_loss': [], 'test_loss': [], 'test_f1': []}
    best_f1_score = 0.0
    
    for epoch in range(exp["epoci"]):
        start_time = time.time()
        
        # --- ANTRENARE ---
        model.train()
        running_train_loss = 0.0
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            
            # Curățare agresivă a memoriei la fiecare batch
            del images, labels, loss
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        # --- VALIDARE ---
        model.eval()
        running_test_loss = 0.0
        
        # Folosim numpy arrays direct, nu liste Python
        num_test_samples = len(test_loader.dataset)
        all_preds = np.zeros(num_test_samples, dtype=np.float32)
        all_labels = np.zeros(num_test_samples, dtype=np.float32)
        current_idx = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                batch_size = images.size(0)
                images = images.to(device)
                labels = labels.to(device).unsqueeze(1)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_test_loss += loss.item()
                
                probs = torch.sigmoid(outputs)
                preds = (probs >= 0.5).float()
                
                # Salvăm în array-ul numpy (foarte eficient ca memorie)
                all_preds[current_idx:current_idx+batch_size] = preds.cpu().numpy().squeeze()
                all_labels[current_idx:current_idx+batch_size] = labels.cpu().numpy().squeeze()
                
                current_idx += batch_size
                
                # Curățare batch
                del images, labels, outputs, loss, probs, preds
                
        avg_test_loss = running_test_loss / len(test_loader)
        epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
        
        # Salvare istoric
        istoric['train_loss'].append(avg_train_loss)
        istoric['test_loss'].append(avg_test_loss)
        istoric['test_f1'].append(epoch_f1)
        
        m, s = divmod(time.time() - start_time, 60)
        
        print(f"  Epoca [{epoch+1}/{exp['epoci']}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_test_loss:.4f} | F1: {epoch_f1:.4f} | {int(m)}m {int(s)}s ", end="")
        
        if epoch_f1 > best_f1_score:
            best_f1_score = epoch_f1
            nume_salvare = f"{exp['nume']}_best.pth"
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), nume_salvare)
            else:
                torch.save(model.state_dict(), nume_salvare)
            print("-> Salvat (Scor record)!")
        else:
            print()
            
        # MODIFICARE CRITICĂ: Forțăm curățarea completă a RAM-ului la finalul fiecărei epoci
        del all_preds, all_labels
        torch.cuda.empty_cache()
        gc.collect()
            
    # 4.5 Generare Grafic
    plt.figure(figsize=(10, 5))
    plt.plot(istoric['train_loss'], label='Train Loss', color='blue')
    plt.plot(istoric['test_loss'], label='Val Loss', color='red', linestyle='--')
    plt.plot(istoric['test_f1'], label='Val F1-Score', color='purple', linestyle='-.')
    plt.title(f"Rezultate: {exp['nume']}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"grafic_{exp['nume']}.png", dpi=300)
    plt.close() # Închidem graficul ca să nu blocheze memoria
    
    # 4.6 CURĂȚARE MEMORIE PENTRU URMĂTORUL EXPERIMENT
    print(f"✔ Experiment complet! Eliberăm memoria...")
    del model
    del optimizer
    del train_loader
    del test_loader
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(2) # O pauză scurtă de siguranță pentru hardware

print("\n🎉 TOATE EXPERIMENTELE AU FOST FINALIZATE CU SUCCES! 🎉")

Rulăm pe: cuda | GPU-uri paralele: 1

🚀 ÎNCEPE EXPERIMENTUL 1/7: Exp1_ResNet50_Combined3_Aug
  -> Bază de date încărcată: 5129 imagini totale.
  Epoca [1/50] | Train Loss: 0.5000 | Val Loss: 0.3929 | F1: 0.8245 | 2m 10s -> Salvat (Scor record)!
  Epoca [2/50] | Train Loss: 0.3928 | Val Loss: 0.3618 | F1: 0.8410 | 3m 3s -> Salvat (Scor record)!
  Epoca [3/50] | Train Loss: 0.3918 | Val Loss: 0.3533 | F1: 0.8427 | 2m 55s -> Salvat (Scor record)!
  Epoca [4/50] | Train Loss: 0.3780 | Val Loss: 0.3652 | F1: 0.8232 | 2m 58s 
  Epoca [5/50] | Train Loss: 0.3647 | Val Loss: 0.3355 | F1: 0.8489 | 2m 56s -> Salvat (Scor record)!
  Epoca [6/50] | Train Loss: 0.3735 | Val Loss: 0.3484 | F1: 0.8333 | 2m 56s 
  Epoca [7/50] | Train Loss: 0.3676 | Val Loss: 0.3568 | F1: 0.8368 | 2m 55s 
  Epoca [8/50] | Train Loss: 0.3804 | Val Loss: 0.3299 | F1: 0.8509 | 2m 52s -> Salvat (Scor record)!
  Epoca [9/50] | Train Loss: 0.3832 | Val Loss: 0.3551 | F1: 0.8321 | 2m 53s 
  Epoca [10/50] | Train Loss: 0.3682

100%|██████████| 30.8M/30.8M [00:00<00:00, 45.5MB/s]


  Epoca [1/50] | Train Loss: 0.5488 | Val Loss: 0.4261 | F1: 0.7968 | 2m 54s -> Salvat (Scor record)!
  Epoca [2/50] | Train Loss: 0.4659 | Val Loss: 0.4081 | F1: 0.7952 | 2m 54s 
  Epoca [3/50] | Train Loss: 0.4380 | Val Loss: 0.3787 | F1: 0.8202 | 2m 50s -> Salvat (Scor record)!
  Epoca [4/50] | Train Loss: 0.4266 | Val Loss: 0.3739 | F1: 0.8269 | 2m 50s -> Salvat (Scor record)!
  Epoca [5/50] | Train Loss: 0.4270 | Val Loss: 0.3552 | F1: 0.8459 | 2m 53s -> Salvat (Scor record)!
  Epoca [6/50] | Train Loss: 0.4216 | Val Loss: 0.3561 | F1: 0.8316 | 2m 52s 
  Epoca [7/50] | Train Loss: 0.4274 | Val Loss: 0.3532 | F1: 0.8317 | 2m 49s 
  Epoca [8/50] | Train Loss: 0.4240 | Val Loss: 0.3529 | F1: 0.8404 | 2m 50s 
  Epoca [9/50] | Train Loss: 0.4219 | Val Loss: 0.3528 | F1: 0.8402 | 2m 51s 
  Epoca [10/50] | Train Loss: 0.4318 | Val Loss: 0.3521 | F1: 0.8443 | 2m 52s 
  Epoca [11/50] | Train Loss: 0.4316 | Val Loss: 0.3484 | F1: 0.8430 | 2m 49s 
  Epoca [12/50] | Train Loss: 0.4325 | Val L

100%|██████████| 20.5M/20.5M [00:00<00:00, 67.5MB/s]


  Epoca [1/50] | Train Loss: 0.5165 | Val Loss: 0.4261 | F1: 0.7979 | 2m 53s -> Salvat (Scor record)!
  Epoca [2/50] | Train Loss: 0.4296 | Val Loss: 0.3926 | F1: 0.8247 | 2m 55s -> Salvat (Scor record)!
  Epoca [3/50] | Train Loss: 0.4250 | Val Loss: 0.3839 | F1: 0.8200 | 2m 52s 
  Epoca [4/50] | Train Loss: 0.3951 | Val Loss: 0.3699 | F1: 0.8340 | 2m 54s -> Salvat (Scor record)!
  Epoca [5/50] | Train Loss: 0.3977 | Val Loss: 0.3602 | F1: 0.8442 | 2m 51s -> Salvat (Scor record)!
  Epoca [6/50] | Train Loss: 0.3908 | Val Loss: 0.3495 | F1: 0.8494 | 2m 52s -> Salvat (Scor record)!
  Epoca [7/50] | Train Loss: 0.3912 | Val Loss: 0.3697 | F1: 0.8286 | 2m 53s 
  Epoca [8/50] | Train Loss: 0.3865 | Val Loss: 0.3655 | F1: 0.8243 | 2m 50s 
  Epoca [9/50] | Train Loss: 0.3958 | Val Loss: 0.3311 | F1: 0.8404 | 2m 57s 
  Epoca [10/50] | Train Loss: 0.3839 | Val Loss: 0.3513 | F1: 0.8345 | 2m 51s 
  Epoca [11/50] | Train Loss: 0.3936 | Val Loss: 0.3500 | F1: 0.8328 | 2m 51s 
  Epoca [12/50] | Tr